In [1]:
# ================================================================
# TB PORTALS MARCH 2025
# NOTEBOOK 05
# FINAL TEMPORAL AND MULTIMODAL PAIR SELECTION
# ================================================================
#
# PROJECT:
# Multimodal Tuberculosis Drug-Resistance Prediction Using
# Chest X-Ray Images and Mycobacterium tuberculosis
# Genomic Features
#
# PURPOSE
# -------
# Construct one final prediction-time CXR + genomic observation
# per eligible condition.
#
# TEMPORAL RULE
# -------------
# Genomic specimen must be collected ON OR BEFORE the selected
# CXR imaging date.
#
# Therefore:
#
#     genomic before CXR -> eligible
#     genomic same day   -> eligible
#     genomic after CXR  -> not eligible for prediction-time pair
#
# THIS NOTEBOOK DOES NOT:
# -----------------------
# - extract images
# - preprocess images
# - resize images
# - apply CLAHE
# - segment lungs
# - engineer genomic ML features
# - train models
# - perform XAI
#
# ================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")


# ================================================================
# 1. PROJECT PATHS
# ================================================================

BASE_DIR = Path(r"C:\TBP")
METADATA_DIR = BASE_DIR / "Metadata"


CXR_SOURCE = (
    METADATA_DIR /
    "TB_Portals_CXRs_March_2025.csv"
)

GENOMIC_SOURCE = (
    METADATA_DIR /
    "TB_Portals_Genomics_March_2025.csv"
)

CXR_MASTER = (
    METADATA_DIR /
    "Step_3B_6_24_Notebook_03" /
    "TB_Portals_March2025_3859_Multimodal_CXR_Master.csv"
)

TEMPORAL_DIR = (
    METADATA_DIR /
    "Step_3B_6_34_CXR_Genomics_Temporal_Reconciliation"
)

TEMPORAL_PAIRS = (
    TEMPORAL_DIR /
    "TB_Portals_March2025_Step_3B_6_34_All_CXR_Genomic_Temporal_Pairs.csv"
)

NEAREST_TEMPORAL = (
    TEMPORAL_DIR /
    "TB_Portals_March2025_Step_3B_6_34_Nearest_Genomic_Timepoint_Per_CXR.csv"
)

GENOMIC_TIMEPOINTS = (
    TEMPORAL_DIR /
    "TB_Portals_March2025_Step_3B_6_34_Unique_Genomic_Timepoints_Common_Cohort.csv"
)


# ================================================================
# 2. OUTPUT DIRECTORY
# ================================================================

OUTPUT_DIR = (
    METADATA_DIR /
    "Step_3B_6_50_Final_Temporal_Multimodal_Pair_Selection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. START
# ================================================================

print("=" * 100)
print("TB PORTALS MARCH 2025")
print("NOTEBOOK 05 — FINAL TEMPORAL AND MULTIMODAL PAIR SELECTION")
print("=" * 100)


# ================================================================
# 4. REQUIRED INPUT VALIDATION
# ================================================================

print("\n1. INPUT FILE VALIDATION")
print("-" * 100)


required_inputs = {
    "CXR source": CXR_SOURCE,
    "Genomic source": GENOMIC_SOURCE,
    "CXR master": CXR_MASTER,
    "Temporal pairs": TEMPORAL_PAIRS,
    "Nearest temporal artifact": NEAREST_TEMPORAL,
    "Genomic timepoints": GENOMIC_TIMEPOINTS
}


for name, path in required_inputs.items():

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired input missing:\n"
            f"{name}\n"
            f"{path}"
        )

    print(
        f"{name:<35}: FOUND"
    )


# ================================================================
# 5. LOAD NOTEBOOK 04 OUTPUT
# ================================================================

NOTEBOOK_04_DIR = (
    METADATA_DIR /
    "Step_3B_6_49_Final_DR_DS_Cohort"
)

NOTEBOOK_04_MASTER = (
    NOTEBOOK_04_DIR /
    "TB_Portals_March2025_Step_3B_6_49_Final_Target_Candidate_Condition_Master.csv"
)

if not NOTEBOOK_04_MASTER.exists():

    raise FileNotFoundError(
        "Notebook 04 condition master was not found:\n"
        f"{NOTEBOOK_04_MASTER}"
    )


condition_master = pd.read_csv(
    NOTEBOOK_04_MASTER,
    low_memory=False
)


print(
    f"\nNotebook 04 condition master rows : "
    f"{len(condition_master):,}"
)


if len(condition_master) != 3081:

    raise ValueError(
        "Notebook 04 condition master must contain "
        "exactly 3,081 conditions."
    )


# ================================================================
# 6. LOAD TEMPORAL DATA
# ================================================================

print("\n2. LOADING TEMPORAL DATA")
print("-" * 100)


temporal = pd.read_csv(
    TEMPORAL_PAIRS,
    low_memory=False
)

nearest_temporal = pd.read_csv(
    NEAREST_TEMPORAL,
    low_memory=False
)

genomic_timepoints = pd.read_csv(
    GENOMIC_TIMEPOINTS,
    low_memory=False
)


print(
    f"Temporal pair rows        : "
    f"{len(temporal):,}"
)

print(
    f"Nearest temporal rows     : "
    f"{len(nearest_temporal):,}"
)

print(
    f"Unique genomic timepoints : "
    f"{len(genomic_timepoints):,}"
)


if len(temporal) != 4146:

    raise ValueError(
        "Expected exactly 4,146 temporal pair records."
    )


if len(genomic_timepoints) != 3260:

    raise ValueError(
        "Expected exactly 3,260 unique genomic timepoints."
    )


# ================================================================
# 7. NORMALIZE IDENTIFIERS
# ================================================================

print("\n3. IDENTIFIER NORMALIZATION")
print("-" * 100)


def normalize_string_column(
    dataframe,
    column
):

    dataframe = dataframe.copy()

    dataframe[column] = (
        dataframe[column]
        .astype("string")
        .str.strip()
    )

    return dataframe


for dataframe in [
    condition_master,
    temporal,
    nearest_temporal,
    genomic_timepoints
]:

    if "condition_id" not in dataframe.columns:

        raise ValueError(
            "condition_id is missing from a required artifact."
        )

    dataframe["condition_id"] = (
        dataframe["condition_id"]
        .astype("string")
        .str.strip()
    )


# ================================================================
# 8. TEMPORAL SCHEMA VALIDATION
# ================================================================

print("\n4. TEMPORAL SCHEMA VALIDATION")
print("-" * 100)


required_temporal_columns = [
    "condition_id",
    "series_instance_content_url",
    "imaging_date_num",
    "specimen_id",
    "specimen_collection_date_num",
    "signed_temporal_delta_days",
    "absolute_temporal_distance_days",
    "temporal_direction"
]


missing_temporal = [
    col
    for col in required_temporal_columns
    if col not in temporal.columns
]


if missing_temporal:

    raise ValueError(
        "Temporal pair artifact is missing required columns:\n"
        f"{missing_temporal}"
    )


print(
    "Temporal schema : PASS"
)


# ================================================================
# 9. DATE / TEMPORAL NUMERIC VALIDATION
# ================================================================

print("\n5. TEMPORAL VALUE VALIDATION")
print("-" * 100)


numeric_temporal_columns = [
    "imaging_date_num",
    "specimen_collection_date_num",
    "signed_temporal_delta_days",
    "absolute_temporal_distance_days"
]


for col in numeric_temporal_columns:

    temporal[col] = pd.to_numeric(
        temporal[col],
        errors="coerce"
    )


for col in numeric_temporal_columns:

    missing_count = int(
        temporal[col].isna().sum()
    )

    print(
        f"{col:<40}: missing = {missing_count}"
    )

    if missing_count > 0:

        raise ValueError(
            f"Temporal column {col} contains missing values."
        )


# ================================================================
# 10. RECONSTRUCT TEMPORAL DIRECTION
# ================================================================
#
# The previous audit demonstrated that the signed temporal
# difference can be independently used to derive direction.
#
# signed_delta = genomic_date - imaging_date
#
# Therefore:
#
#     negative -> genomic before CXR
#     zero     -> same day
#     positive -> genomic after CXR
#
# ================================================================

print("\n6. INDEPENDENT TEMPORAL DIRECTION VALIDATION")
print("-" * 100)


def derive_direction(delta):

    if delta < 0:

        return "GENOMIC_BEFORE_CXR"

    elif delta == 0:

        return "SAME_DAY"

    else:

        return "GENOMIC_AFTER_CXR"


temporal[
    "derived_temporal_direction"
] = (
    temporal[
        "signed_temporal_delta_days"
    ]
    .apply(
        derive_direction
    )
)


direction_comparison = (
    temporal[
        [
            "temporal_direction",
            "derived_temporal_direction"
        ]
    ]
    .value_counts()
    .reset_index(
        name="pair_count"
    )
)


print(
    direction_comparison.to_string(
        index=False
    )
)


# ================================================================
# 11. CHECK STORED DIRECTION CONTENT
# ================================================================

print("\n7. STORED VS DERIVED TEMPORAL DIRECTION")
print("-" * 100)


stored_normalized = (
    temporal[
        "temporal_direction"
    ]
    .astype("string")
    .str.strip()
)


temporal[
    "stored_direction_normalized"
] = stored_normalized


direction_mapping = {

    "GENOMICS_BEFORE_CXR":
        "GENOMIC_BEFORE_CXR",

    "GENOMICS_AFTER_CXR":
        "GENOMIC_AFTER_CXR",

    "SAME_DAY":
        "SAME_DAY"
}


temporal[
    "normalized_stored_direction"
] = (
    temporal[
        "stored_direction_normalized"
    ]
    .map(direction_mapping)
)


unknown_stored_direction = (
    temporal[
        "normalized_stored_direction"
    ]
    .isna()
)


if unknown_stored_direction.any():

    unknown_values = (
        temporal.loc[
            unknown_stored_direction,
            "stored_direction_normalized"
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Unknown stored temporal direction values:\n"
        f"{unknown_values}"
    )


direction_match = (
    temporal[
        "normalized_stored_direction"
    ]
    ==
    temporal[
        "derived_temporal_direction"
    ]
)


print(
    f"Direction agreement : "
    f"{int(direction_match.sum()):,} / {len(temporal):,}"
)


if not direction_match.all():

    raise ValueError(
        "Stored and independently derived temporal directions "
        "do not agree."
    )


print(
    "Temporal direction validation : PASS"
)


# ================================================================
# 12. FULL TEMPORAL DISTRIBUTION
# ================================================================

print("\n8. FULL TEMPORAL DISTRIBUTION")
print("-" * 100)


full_temporal_distribution = (
    temporal[
        "derived_temporal_direction"
    ]
    .value_counts()
    .rename_axis(
        "temporal_direction"
    )
    .reset_index(
        name="pair_count"
    )
)


full_temporal_distribution[
    "pair_percentage"
] = (
    full_temporal_distribution[
        "pair_count"
    ]
    /
    len(temporal)
    *
    100
)


print(
    full_temporal_distribution.to_string(
        index=False
    )
)


# ================================================================
# 13. DEFINE PREDICTION-TIME ELIGIBILITY
# ================================================================
#
# PRIMARY RULE
# ------------
# Genomic specimen must be available ON OR BEFORE the CXR.
#
# Eligible:
#     signed delta <= 0
#
# Not eligible:
#     signed delta > 0
#
# ================================================================

print("\n9. PREDICTION-TIME TEMPORAL ELIGIBILITY")
print("-" * 100)


temporal[
    "prediction_time_eligible"
] = (
    temporal[
        "signed_temporal_delta_days"
    ]
    <= 0
)


temporal[
    "prediction_temporal_class"
] = np.select(

    [
        temporal[
            "signed_temporal_delta_days"
        ] < 0,

        temporal[
            "signed_temporal_delta_days"
        ] == 0
    ],

    [
        "GENOMIC_BEFORE_CXR",
        "SAME_DAY"
    ],

    default="GENOMIC_AFTER_CXR"
)


eligible_pairs = temporal[
    temporal[
        "prediction_time_eligible"
    ]
].copy()


ineligible_after_pairs = temporal[
    ~temporal[
        "prediction_time_eligible"
    ]
].copy()


print(
    f"Total temporal pairs       : "
    f"{len(temporal):,}"
)

print(
    f"Eligible before/same-day    : "
    f"{len(eligible_pairs):,}"
)

print(
    f"Post-CXR pairs excluded     : "
    f"{len(ineligible_after_pairs):,}"
)


# ================================================================
# 14. CONDITION-LEVEL TEMPORAL AVAILABILITY
# ================================================================

print("\n10. CONDITION-LEVEL TEMPORAL AVAILABILITY")
print("-" * 100)


condition_temporal_availability = (
    temporal
    .groupby("condition_id")
    .agg(

        total_temporal_pairs=(
            "condition_id",
            "size"
        ),

        eligible_temporal_pairs=(
            "prediction_time_eligible",
            "sum"
        ),

        post_cxr_pairs=(
            "prediction_time_eligible",
            lambda x:
                int(
                    (~x).sum()
                )
        ),

        minimum_absolute_distance_days=(
            "absolute_temporal_distance_days",
            "min"
        ),

        minimum_signed_delta_days=(
            "signed_temporal_delta_days",
            "min"
        ),

        maximum_signed_delta_days=(
            "signed_temporal_delta_days",
            "max"
        )
    )
    .reset_index()
)


condition_temporal_availability[
    "has_prediction_time_genomics"
] = (
    condition_temporal_availability[
        "eligible_temporal_pairs"
    ]
    > 0
)


print(
    "Conditions with prediction-time genomic availability:",
    int(
        condition_temporal_availability[
            "has_prediction_time_genomics"
        ].sum()
    )
)


print(
    "Conditions without prediction-time genomic availability:",
    int(
        (
            ~condition_temporal_availability[
                "has_prediction_time_genomics"
            ]
        ).sum()
    )
)


# ================================================================
# 15. CXR HUMAN-REVIEW EVIDENCE DISCOVERY
# ================================================================
#
# We discover the final semantic/structural reconciliation file
# rather than assuming that its exact folder name is unchanged.
#
# ================================================================

print("\n11. IMAGE HUMAN-REVIEW EVIDENCE DISCOVERY")
print("-" * 100)


candidate_review_files = list(
    METADATA_DIR.rglob(
        "TB_Portals_March2025_Step_3B_6_23U_Final_Semantic_Structural_Reconciliation.csv"
    )
)


if len(candidate_review_files) == 0:

    print(
        "Final 23U reconciliation file not found."
    )

    print(
        "The selection stage will proceed without applying "
        "human-review exclusions."
    )

    review = None

elif len(candidate_review_files) > 1:

    raise RuntimeError(
        "Multiple 23U reconciliation files found. "
        "Ambiguous input; stop rather than guessing:\n"
        +
        "\n".join(
            str(p)
            for p in candidate_review_files
        )
    )

else:

    review_path = candidate_review_files[0]

    print(
        f"Review artifact found:\n{review_path}"
    )

    review = pd.read_csv(
        review_path,
        low_memory=False
    )

    print(
        f"Review rows : {len(review):,}"
    )


# ================================================================
# 16. BUILD IMAGE EXCLUSION MAP
# ================================================================

print("\n12. HUMAN-REVIEW IMAGE EXCLUSION MAP")
print("-" * 100)


if review is not None:

    print(
        "Review columns:"
    )

    for col in review.columns:

        print(
            f"  {col}"
        )

    # --------------------------------------------
    # Identify image URL column
    # --------------------------------------------

    possible_url_columns = [
        "series_instance_content_url",
        "image_path",
        "url",
        "cxr_url"
    ]

    review_url_column = next(
        (
            c
            for c in possible_url_columns
            if c in review.columns
        ),
        None
    )

    if review_url_column is None:

        print(
            "No direct CXR URL column found in 23U artifact."
        )

        print(
            "Human-review evidence will not be used for "
            "automatic pair rejection in this notebook."
        )

        review = None

    else:

        review[
            "_review_url"
        ] = (
            review[
                review_url_column
            ]
            .astype("string")
            .str.strip()
        )

        # --------------------------------------------
        # Identify final decision column
        # --------------------------------------------

        possible_decision_columns = [
            "final_decision",
            "decision",
            "semantic_decision",
            "human_decision",
            "resolution"
        ]

        review_decision_column = next(
            (
                c
                for c in possible_decision_columns
                if c in review.columns
            ),
            None
        )

        if review_decision_column is None:

            print(
                "No final decision column found."
            )

            print(
                "Human-review evidence will not be used "
                "for automatic pair rejection."
            )

            review = None

        else:

            review[
                "_review_decision"
            ] = (
                review[
                    review_decision_column
                ]
                .astype("string")
                .str.strip()
                .str.upper()
            )

            excluded_urls = set(
                review.loc[
                    review[
                        "_review_decision"
                    ] == "EXCLUDE",
                    "_review_url"
                ]
                .dropna()
                .tolist()
            )

            print(
                f"Explicitly excluded reviewed images : "
                f"{len(excluded_urls):,}"
            )


# ================================================================
# 17. IMAGE EXCLUSION FLAG
# ================================================================

if review is None:

    temporal[
        "human_review_excluded"
    ] = False

else:

    temporal[
        "human_review_excluded"
    ] = (
        temporal[
            "series_instance_content_url"
        ]
        .astype("string")
        .str.strip()
        .isin(
            excluded_urls
        )
    )


print(
    f"Temporal rows linked to explicitly excluded images: "
    f"{int(temporal['human_review_excluded'].sum()):,}"
)


# ================================================================
# 18. ELIGIBLE PAIR CANDIDATES
# ================================================================

print("\n13. BUILDING ELIGIBLE PAIR CANDIDATES")
print("-" * 100)


pair_candidates = temporal[
    (
        temporal[
            "prediction_time_eligible"
        ]
    )
    &
    (
        ~temporal[
            "human_review_excluded"
        ]
    )
].copy()


print(
    f"Prediction-time eligible pairs      : "
    f"{int(temporal['prediction_time_eligible'].sum()):,}"
)

print(
    f"Eligible after reviewed-image filter: "
    f"{len(pair_candidates):,}"
)


# ================================================================
# 19. CONDITION-LEVEL ELIGIBLE PAIR AVAILABILITY
# ================================================================

eligible_condition_counts = (
    pair_candidates[
        "condition_id"
    ]
    .nunique()
)


print(
    f"Conditions with at least one "
    f"eligible pair : {eligible_condition_counts:,}"
)


candidate_conditions = set(
    condition_master[
        "condition_id"
    ]
)


eligible_conditions = set(
    pair_candidates[
        "condition_id"
    ]
)


no_eligible_pair_conditions = (
    candidate_conditions
    -
    eligible_conditions
)


print(
    f"Conditions without an eligible pair : "
    f"{len(no_eligible_pair_conditions):,}"
)


# ================================================================
# 20. REPRESENTATIVE PAIR SELECTION
# ================================================================
#
# Selection hierarchy:
#
# 1. prediction-time eligible
# 2. explicitly reviewed EXCLUDE images removed
# 3. smallest absolute temporal distance
# 4. same-day preferred naturally because distance = 0
# 5. deterministic specimen date
# 6. deterministic CXR URL
# 7. deterministic specimen ID
#
# This guarantees reproducibility.
#
# ================================================================

print("\n14. REPRESENTATIVE MULTIMODAL PAIR SELECTION")
print("-" * 100)


pair_candidates[
    "_abs_delta"
] = (
    pair_candidates[
        "absolute_temporal_distance_days"
    ]
)


pair_candidates[
    "_specimen_date_sort"
] = (
    pair_candidates[
        "specimen_collection_date_num"
    ]
)


pair_candidates[
    "_cxr_date_sort"
] = (
    pair_candidates[
        "imaging_date_num"
    ]
)


pair_candidates[
    "_cxr_url_sort"
] = (
    pair_candidates[
        "series_instance_content_url"
    ]
    .astype("string")
)


pair_candidates[
    "_specimen_id_sort"
] = (
    pair_candidates[
        "specimen_id"
    ]
    .astype("string")
)


pair_candidates = (
    pair_candidates
    .sort_values(
        by=[
            "condition_id",
            "_abs_delta",
            "_specimen_date_sort",
            "_cxr_date_sort",
            "_cxr_url_sort",
            "_specimen_id_sort"
        ],
        ascending=[
            True,
            True,
            False,
            False,
            True,
            True
        ],
        kind="mergesort"
    )
)


selected_pairs = (
    pair_candidates
    .drop_duplicates(
        subset=[
            "condition_id"
        ],
        keep="first"
    )
    .copy()
)


print(
    f"Selected representative pairs : "
    f"{len(selected_pairs):,}"
)


# ================================================================
# 21. UNIQUE CONDITION VALIDATION
# ================================================================

if (
    selected_pairs[
        "condition_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Representative pair selection produced "
        "duplicate conditions."
    )


# ================================================================
# 22. TEMPORAL DIRECTION OF FINAL PAIRS
# ================================================================

print("\n15. FINAL REPRESENTATIVE TEMPORAL DIRECTION")
print("-" * 100)


final_temporal_distribution = (
    selected_pairs[
        "prediction_temporal_class"
    ]
    .value_counts()
    .rename_axis(
        "temporal_class"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    final_temporal_distribution.to_string(
        index=False
    )
)


# ================================================================
# 23. FINAL TIME DISTANCE DISTRIBUTION
# ================================================================

print("\n16. FINAL TEMPORAL DISTANCE")
print("-" * 100)


distance_series = selected_pairs[
    "absolute_temporal_distance_days"
].astype(float)


distance_statistics = pd.DataFrame({
    "statistic": [
        "count",
        "mean_days",
        "median_days",
        "minimum_days",
        "maximum_days",
        "p25_days",
        "p75_days",
        "p90_days",
        "p95_days",
        "p99_days"
    ],

    "value": [

        len(distance_series),

        distance_series.mean(),

        distance_series.median(),

        distance_series.min(),

        distance_series.max(),

        distance_series.quantile(0.25),

        distance_series.quantile(0.75),

        distance_series.quantile(0.90),

        distance_series.quantile(0.95),

        distance_series.quantile(0.99)
    ]
})


print(
    distance_statistics.to_string(
        index=False
    )
)


# ================================================================
# 24. MERGE TARGET INFORMATION
# ================================================================

print("\n17. ATTACHING DR-TB / DS-TB TARGET")
print("-" * 100)


final_pairs = (
    selected_pairs
    .merge(
        condition_master[
            [
                "condition_id",
                "cxr_resistance",
                "genomic_resistance",
                "target_class",
                "target_binary",
                "cxr_record_count",
                "cxr_url_count",
                "multiple_cxr",
                "genomic_record_count",
                "genomic_drug_resistance_nunique"
            ]
        ],
        on="condition_id",
        how="left",
        validate="one_to_one"
    )
)


if len(final_pairs) != len(selected_pairs):

    raise RuntimeError(
        "Target merge changed the number of selected pairs."
    )


if final_pairs[
    "target_class"
].isna().any():

    raise RuntimeError(
        "Some selected pairs are missing target labels."
    )


# ================================================================
# 25. VERIFY RESISTANCE AGREEMENT AGAIN
# ================================================================

final_pairs[
    "target_label_agreement"
] = (
    final_pairs[
        "cxr_resistance"
    ]
    ==
    final_pairs[
        "genomic_resistance"
    ]
)


if not final_pairs[
    "target_label_agreement"
].all():

    raise RuntimeError(
        "CXR/genomic resistance disagreement detected "
        "inside final pair cohort."
    )


print(
    "CXR/genomic resistance agreement : PASS"
)


# ================================================================
# 26. FINAL TARGET DISTRIBUTION
# ================================================================

print("\n18. FINAL PREDICTION-TIME TARGET DISTRIBUTION")
print("-" * 100)


final_target_distribution = (
    final_pairs[
        "target_class"
    ]
    .value_counts()
    .rename_axis(
        "target_class"
    )
    .reset_index(
        name="condition_count"
    )
)


final_target_distribution[
    "percentage"
] = (
    final_target_distribution[
        "condition_count"
    ]
    /
    len(final_pairs)
    *
    100
)


print(
    final_target_distribution.to_string(
        index=False
    )
)


# ================================================================
# 27. CLASS-SPECIFIC TEMPORAL AVAILABILITY
# ================================================================

print("\n19. CLASS-SPECIFIC TEMPORAL AVAILABILITY")
print("-" * 100)


class_temporal_summary = (
    final_pairs
    .groupby(
        [
            "target_class",
            "prediction_temporal_class"
        ]
    )
    .agg(
        condition_count=(
            "condition_id",
            "nunique"
        )
    )
    .reset_index()
)


print(
    class_temporal_summary.to_string(
        index=False
    )
)


# ================================================================
# 28. IMAGE MULTIPLICITY IN FINAL COHORT
# ================================================================

print("\n20. IMAGE MULTIPLICITY IN FINAL COHORT")
print("-" * 100)


final_image_multiplicity = (
    final_pairs[
        "cxr_url_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "number_of_cxr_images_for_condition"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    final_image_multiplicity.to_string(
        index=False
    )
)


# ================================================================
# 29. GENOMIC MULTIPLICITY IN FINAL COHORT
# ================================================================

print("\n21. GENOMIC MULTIPLICITY IN FINAL COHORT")
print("-" * 100)


final_genomic_multiplicity = (
    final_pairs[
        "genomic_record_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "number_of_genomic_records_for_condition"
    )
    .reset_index(
        name="condition_count"
    )
)


print(
    final_genomic_multiplicity.to_string(
        index=False
    )
)


# ================================================================
# 30. CONDITIONS LOST DURING TEMPORAL SELECTION
# ================================================================

print("\n22. COHORT REDUCTION AUDIT")
print("-" * 100)


candidate_condition_set = set(
    condition_master[
        "condition_id"
    ]
)

final_condition_set = set(
    final_pairs[
        "condition_id"
    ]
)


excluded_conditions = (
    candidate_condition_set
    -
    final_condition_set
)


excluded_condition_audit = (
    condition_master[
        condition_master[
            "condition_id"
        ].isin(
            excluded_conditions
        )
    ]
    .copy()
)


excluded_condition_audit[
    "temporal_selection_status"
] = "EXCLUDED_FROM_FINAL_PREDICTION_TIME_COHORT"


excluded_condition_audit[
    "temporal_exclusion_reason"
] = "NO_ELIGIBLE_CXR_GENOMIC_PAIR"


excluded_condition_audit = (
    excluded_condition_audit
    .merge(
        condition_temporal_availability[
            [
                "condition_id",
                "total_temporal_pairs",
                "eligible_temporal_pairs",
                "post_cxr_pairs",
                "minimum_absolute_distance_days",
                "minimum_signed_delta_days",
                "maximum_signed_delta_days"
            ]
        ],
        on="condition_id",
        how="left",
        validate="one_to_one"
    )
)


print(
    f"Candidate conditions          : "
    f"{len(candidate_condition_set):,}"
)

print(
    f"Final prediction-time cohort  : "
    f"{len(final_condition_set):,}"
)

print(
    f"Conditions excluded           : "
    f"{len(excluded_conditions):,}"
)


# ================================================================
# 31. DETERMINE WHY EXCLUDED CONDITIONS HAVE NO PAIR
# ================================================================

if len(excluded_condition_audit) > 0:

    excluded_condition_audit[
        "all_temporal_pairs_after_cxr"
    ] = (
        excluded_condition_audit[
            "post_cxr_pairs"
        ]
        ==
        excluded_condition_audit[
            "total_temporal_pairs"
        ]
    )

    excluded_condition_audit[
        "no_temporal_pair"
    ] = (
        excluded_condition_audit[
            "total_temporal_pairs"
        ].fillna(0)
        ==
        0
    )

    excluded_condition_audit[
        "reviewed_images_excluded_only"
    ] = False

    # Determine if temporal pairs existed but all were rejected
    # by explicit human image exclusion.

    for condition_id in excluded_condition_audit[
        "condition_id"
    ]:

        condition_temporal_rows = temporal[
            temporal[
                "condition_id"
            ]
            ==
            condition_id
        ]

        eligible_before_review = (
            condition_temporal_rows[
                "signed_temporal_delta_days"
            ]
            <= 0
        )

        if (
            eligible_before_review.any()
            and
            condition_temporal_rows.loc[
                eligible_before_review,
                "human_review_excluded"
            ].all()
        ):

            excluded_condition_audit.loc[
                excluded_condition_audit[
                    "condition_id"
                ]
                ==
                condition_id,
                "reviewed_images_excluded_only"
            ] = True


    excluded_condition_audit[
        "temporal_exclusion_reason"
    ] = np.select(

        [
            excluded_condition_audit[
                "no_temporal_pair"
            ],

            excluded_condition_audit[
                "all_temporal_pairs_after_cxr"
            ],

            excluded_condition_audit[
                "reviewed_images_excluded_only"
            ]
        ],

        [
            "NO_TEMPORAL_CXR_GENOMIC_PAIR",

            "ALL_GENOMIC_SAMPLES_AFTER_CXR",

            "ALL_PREDICTION_TIME_PAIRS_USED_EXPLICITLY_EXCLUDED_IMAGES"
        ],

        default="NO_ELIGIBLE_PAIR_AFTER_SELECTION_RULES"
    )


# ================================================================
# 32. FINAL COHORT INTEGRITY
# ================================================================

print("\n23. FINAL COHORT INTEGRITY")
print("-" * 100)


integrity_checks = {

    "One row per condition":
        not final_pairs[
            "condition_id"
        ].duplicated().any(),

    "No missing target":
        not final_pairs[
            "target_class"
        ].isna().any(),

    "No missing CXR URL":
        not final_pairs[
            "series_instance_content_url"
        ].isna().any(),

    "No missing specimen ID":
        not final_pairs[
            "specimen_id"
        ].isna().any(),

    "All genomic dates <= CXR dates":
        (
            final_pairs[
                "signed_temporal_delta_days"
            ]
            <= 0
        ).all(),

    "No selected human-excluded image":
        (
            ~final_pairs[
                "human_review_excluded"
            ]
        ).all(),

    "CXR/genomic target agreement":
        final_pairs[
            "target_label_agreement"
        ].all(),

    "Final cohort subset of 3,081":
        set(
            final_pairs[
                "condition_id"
            ]
        ).issubset(
            candidate_condition_set
        )
}


for check, result in integrity_checks.items():

    print(
        f"{check:<50}: "
        f"{'PASS' if result else 'FAIL'}"
    )


if not all(
    integrity_checks.values()
):

    raise RuntimeError(
        "Final cohort integrity validation FAILED. "
        "Do not proceed."
    )


# ================================================================
# 33. CREATE CLEAN FINAL COHORT TABLE
# ================================================================

print("\n24. BUILDING CLEAN FINAL MULTIMODAL COHORT")
print("-" * 100)


final_cohort_columns = [

    "condition_id",

    "patient_id",

    "series_instance_content_url",

    "specimen_id",

    "imaging_date_num",

    "specimen_collection_date_num",

    "signed_temporal_delta_days",

    "absolute_temporal_distance_days",

    "prediction_temporal_class",

    "target_class",

    "target_binary",

    "cxr_resistance",

    "genomic_resistance",

    "cxr_record_count",

    "cxr_url_count",

    "multiple_cxr",

    "genomic_record_count",

    "genomic_drug_resistance_nunique",

    "human_review_excluded"
]


available_final_columns = [
    c
    for c in final_cohort_columns
    if c in final_pairs.columns
]


final_cohort = (
    final_pairs[
        available_final_columns
    ]
    .copy()
)


# ================================================================
# 34. FINAL COHORT HASH
# ================================================================

print("\n25. FINAL COHORT REPRODUCIBILITY HASH")
print("-" * 100)


hash_columns = [
    "condition_id",
    "series_instance_content_url",
    "specimen_id",
    "imaging_date_num",
    "specimen_collection_date_num",
    "target_class"
]


hash_table = (
    final_cohort[
        hash_columns
    ]
    .sort_values(
        by=[
            "condition_id"
        ],
        kind="mergesort"
    )
    .fillna("")
)


hash_string = (
    hash_table
    .astype(str)
    .to_csv(
        index=False,
        lineterminator="\n"
    )
)


cohort_sha256 = hashlib.sha256(
    hash_string.encode(
        "utf-8"
    )
).hexdigest()


print(
    f"Final cohort SHA-256:\n{cohort_sha256}"
)


# ================================================================
# 35. SAVE FINAL COHORT
# ================================================================

print("\n26. SAVING FINAL COHORT OUTPUT")
print("-" * 100)


final_cohort_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Final_Prediction_Time_Multimodal_Cohort.csv"
)


final_cohort.to_csv(
    final_cohort_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"{final_cohort_path.name}: PASS"
)


# ================================================================
# 36. SAVE EXCLUSION AUDIT
# ================================================================

excluded_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Temporal_Exclusion_Audit.csv"
)


excluded_condition_audit.to_csv(
    excluded_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"{excluded_path.name}: PASS"
)


# ================================================================
# 37. SAVE TEMPORAL DISTRIBUTION
# ================================================================

temporal_distribution_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Final_Temporal_Distribution.csv"
)


final_temporal_distribution.to_csv(
    temporal_distribution_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"{temporal_distribution_path.name}: PASS"
)


# ================================================================
# 38. SAVE TARGET DISTRIBUTION
# ================================================================

target_distribution_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Final_Target_Distribution.csv"
)


final_target_distribution.to_csv(
    target_distribution_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"{target_distribution_path.name}: PASS"
)


# ================================================================
# 39. SAVE TEMPORAL DISTANCE STATISTICS
# ================================================================

distance_statistics_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Temporal_Distance_Statistics.csv"
)


distance_statistics.to_csv(
    distance_statistics_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"{distance_statistics_path.name}: PASS"
)


# ================================================================
# 40. SAVE CLASS/TEMPORAL SUMMARY
# ================================================================

class_temporal_summary_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Class_Temporal_Summary.csv"
)


class_temporal_summary.to_csv(
    class_temporal_summary_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"{class_temporal_summary_path.name}: PASS"
)


# ================================================================
# 41. SAVE DECISION RECORD
# ================================================================

decision_record = {

    "project":
        "Multimodal Tuberculosis Drug-Resistance Prediction "
        "Using Chest X-Ray Images and Mycobacterium tuberculosis "
        "Genomic Features",

    "notebook":
        "05_Final_Temporal_and_Multimodal_Pair_Selection",

    "case_unit":
        "condition_id",

    "candidate_conditions":
        len(candidate_condition_set),

    "final_prediction_time_conditions":
        len(final_condition_set),

    "excluded_conditions":
        len(excluded_conditions),

    "temporal_rule":
        "Genomic specimen collected on or before CXR date",

    "eligible_temporal_directions":
        [
            "GENOMIC_BEFORE_CXR",
            "SAME_DAY"
        ],

    "ineligible_temporal_direction":
        "GENOMIC_AFTER_CXR",

    "representative_pair_rule":
        "Minimum absolute temporal distance among eligible "
        "prediction-time pairs, after exclusion of explicitly "
        "human-adjudicated image exclusions",

    "tie_breaking":
        [
            "minimum absolute temporal distance",
            "latest genomic specimen date",
            "latest CXR date",
            "lexicographically smallest CXR URL",
            "lexicographically smallest specimen ID"
        ],

    "target":
        "DR-TB vs DS-TB",

    "ds_definition":
        "Sensitive",

    "dr_definition":
        "All explicitly classified non-Sensitive resistance categories",

    "direct_target_fields_prohibited_as_predictors":
        [
            "type_of_resistance",
            "drug_resistance_type"
        ],

    "preprocessing_started":
        False,

    "lung_segmentation_started":
        False,

    "model_training_started":
        False,

    "xai_started":
        False,

    "raw_dicom_modified":
        False,

    "source_metadata_modified":
        False,

    "cohort_sha256":
        cohort_sha256
}


decision_path = (
    OUTPUT_DIR /
    "TB_Portals_March2025_Step_3B_6_50_Decision_Record.json"
)


with open(
    decision_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        decision_record,
        f,
        indent=4
    )


print(
    f"{decision_path.name}: PASS"
)


# ================================================================
# 42. FINAL VALIDATION
# ================================================================

print("\n27. FINAL VALIDATION")
print("-" * 100)


final_validation = {

    "Temporal input = 4,146":
        len(temporal) == 4146,

    "Candidate conditions = 3,081":
        len(candidate_condition_set) == 3081,

    "One final row per condition":
        not final_cohort[
            "condition_id"
        ].duplicated().any(),

    "No duplicate CXR URL":
        not final_cohort[
            "series_instance_content_url"
        ].duplicated().any(),

    "No duplicate specimen within final pair":
        not final_cohort[
            "specimen_id"
        ].duplicated().any(),

    "All genomic samples available by CXR":
        (
            final_cohort[
                "signed_temporal_delta_days"
            ]
            <= 0
        ).all(),

    "All target labels present":
        not final_cohort[
            "target_class"
        ].isna().any(),

    "All target labels agree":
        final_pairs[
            "target_label_agreement"
        ].all(),

    "No selected reviewed-excluded image":
        (
            ~final_cohort[
                "human_review_excluded"
            ]
        ).all(),

    "Raw DICOM unchanged":
        True,

    "Source metadata unchanged":
        True,

    "No preprocessing":
        True,

    "No segmentation":
        True,

    "No model training":
        True,

    "No XAI":
        True
}


all_pass = True


for check, result in final_validation.items():

    print(
        f"{check:<55}: "
        f"{'PASS' if result else 'FAIL'}"
    )

    if not result:

        all_pass = False


if not all_pass:

    raise RuntimeError(
        "NOTEBOOK 05 VALIDATION FAILED. "
        "Do not proceed to Notebook 06."
    )


# ================================================================
# 43. FINAL STATUS
# ================================================================

print()
print("=" * 100)
print("NOTEBOOK 05 — FINAL STATUS")
print("=" * 100)

print(
    f"Candidate multimodal conditions : "
    f"{len(candidate_condition_set):,}"
)

print(
    f"Final prediction-time cohort    : "
    f"{len(final_cohort):,}"
)

print(
    f"Excluded conditions             : "
    f"{len(excluded_conditions):,}"
)

print(
    f"DR-TB in final cohort           : "
    f"{int((final_cohort['target_class'] == 'DR-TB').sum()):,}"
)

print(
    f"DS-TB in final cohort           : "
    f"{int((final_cohort['target_class'] == 'DS-TB').sum()):,}"
)

print(
    "Temporal rule                   : LOCKED"
)

print(
    "Representative pair rule       : LOCKED"
)

print(
    "Prediction-time cohort         : LOCKED"
)

print(
    "Image preprocessing             : NOT STARTED"
)

print(
    "Lung segmentation               : NOT STARTED"
)

print(
    "Genomic feature engineering     : NOT STARTED"
)

print(
    "Model training                  : NOT STARTED"
)

print(
    "XAI                             : NOT STARTED"
)

print(
    "Technical validation            : PASS"
)

print("=" * 100)
print("NOTEBOOK 05 COMPLETE")
print("=" * 100)

TB PORTALS MARCH 2025
NOTEBOOK 05 — FINAL TEMPORAL AND MULTIMODAL PAIR SELECTION

1. INPUT FILE VALIDATION
----------------------------------------------------------------------------------------------------
CXR source                         : FOUND
Genomic source                     : FOUND
CXR master                         : FOUND
Temporal pairs                     : FOUND
Nearest temporal artifact          : FOUND
Genomic timepoints                 : FOUND

Notebook 04 condition master rows : 3,081

2. LOADING TEMPORAL DATA
----------------------------------------------------------------------------------------------------
Temporal pair rows        : 4,146
Nearest temporal rows     : 3,871
Unique genomic timepoints : 3,260

3. IDENTIFIER NORMALIZATION
----------------------------------------------------------------------------------------------------

4. TEMPORAL SCHEMA VALIDATION
----------------------------------------------------------------------------------------------------
